# SQ2 — Amanda Luijendijk
**Question:** How is awareness of feed personalisation related to perceived control, and does this relationship differ across age groups?

I look at whether people who know their feed is shaped by personal information feel more in control. I also check whether this link differs by age. Awareness is not the same as the amount of personalisation someone receives.

My main steps are data preparation, chi-square, regression, SEM and clustering. Each step has a short explanation and a simple figure. Ranking, scaling and extra checks are in the appendix. The results show links in survey answers, not cause and effect. Prepared with Codex assistance; I need to understand and explain the choices before submission.


## 1 Prepare the data
I keep the source unchanged and turn agreement answers into scores from 1 (strongly disagree) to 5 (strongly agree). I keep age as categories. Each analysis leaves out only the answers it needs, so the sample sizes differ.

The two main questions are awareness of personal-data-based feed shaping and “I feel I am in control of my feed”. The two extra control items are used separately for clustering and as a candidate factor in SEM. “The algorithm decides what I watch” is reversed as 6 minus the score.

Place Responses.xlsx beside this notebook, set BS_DATA_PATH, or mount the shared Google Drive folder. Install requirements.txt once if needed.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from IPython.display import display, Markdown

paths = [Path(os.getenv('BS_DATA_PATH', 'Responses.xlsx')),
         Path('/content/drive/MyDrive/Colab Notebooks/Responses.xlsx'),
         Path.home() / 'Downloads' / 'Responses (3).xlsx']
file = next((p for p in paths if p.is_file()), None)
if file is None:
    raise FileNotFoundError('Add Responses.xlsx or set BS_DATA_PATH.')
raw = pd.read_excel(file)
raw.columns = raw.columns.str.replace('\u00a0', ' ', regex=False).str.strip()
scale = {'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5}
questions = {
    'awareness': 'I am informed that my feed is shaped by personal information like my age, gender or location - transparency',
    'control': 'I feel I am in control of my feed',
    'influence': 'I can influence the type of content I get on my feed',
    'less_algorithm_control': 'The algorithm decides what I watch'}
df = pd.DataFrame(index=raw.index)
for name, question in questions.items():
    answers = raw[question].astype('string').str.strip().replace('', pd.NA)
    assert answers.dropna().isin(scale).all(), f'Unknown labels in {name}'
    df[name] = answers.map(scale).astype(float)
df['less_algorithm_control'] = 6 - df['less_algorithm_control']
df['age'] = raw['What age group are you in?'].map(lambda x: x.strip() if isinstance(x, str) else x)
df['social_use'] = raw['I use social media'].map(lambda x: x.strip() if isinstance(x, str) else x)
age_order = ['Under 18', '18-24', '25-34', '35-44', '45-54', '55-64', '65 and older']
assert df.age.dropna().isin(age_order).all(), 'Unknown age category'
sq2 = df[['awareness', 'control', 'age']].dropna()
print('Survey rows:', len(raw), '| Main analysis:', len(sq2))
display(df[['awareness', 'control', 'age']].isna().sum().to_frame('Missing answers'))


## 2 Chi square with three answer groups
I first compare awareness and control without including age. For this test only, I use **Disagree (1–2), Neutral (3), Agree (4–5)**.

**H0:** The two answer categories are independent. **H1:** They are associated. I use 0.05 as the significance level. The bars show percentages within each awareness category. The code checks expected counts before using the usual chi-square approximation. This is an exploratory independence test, not the SEM model-fit test.


In [ ]:
three = {1: 'Disagree', 2: 'Disagree', 3: 'Neutral', 4: 'Agree', 5: 'Agree'}
pair = df[['awareness', 'control']].dropna()
order = ['Disagree', 'Neutral', 'Agree']
table = pd.crosstab(pair.awareness.map(three), pair.control.map(three)).reindex(index=order, columns=order, fill_value=0)
chi2, chi_p, chi_dof, expected = stats.chi2_contingency(table, correction=False)
if expected.min() < 1 or (expected < 5).mean() > .20:
    raise ValueError('Expected counts are too small; use a permutation test.')
cramer_v = np.sqrt(chi2 / (len(pair) * min(table.shape[0]-1, table.shape[1]-1)))
display(table)
print(f'N={len(pair)}, chi-square({chi_dof})={chi2:.3f}, p={chi_p:.6f}, V={cramer_v:.3f}')
print('Smallest expected count:', round(expected.min(), 2))
ax = table.div(table.sum(axis=1), axis=0).mul(100).plot.bar(rot=0, figsize=(7, 4))
ax.set(xlabel='Awareness category', ylabel='Percentage within category', ylim=(0, 100), title='Awareness and control')
ax.legend(title='Control', ncol=3); plt.tight_layout(); plt.show()
display(Markdown('We do not reject H0: there is no clear evidence of an association at 0.05.' if chi_p >= .05 else 'We reject H0: there is evidence of an association at 0.05.'))


## 3 Awareness control and age
The regression includes awareness and age group. **H0:** The awareness coefficient is zero. **H1:** It is nonzero. The extra interaction test asks whether the awareness slope is the same across all age groups.

Regression treats the five score steps as equally spaced. I use HC3 standard errors to allow unequal variation. The dot below is the awareness coefficient; the line is its 95% confidence interval. If that line includes zero, there is no clear overall linear link at 0.05. The appendix shows age-specific slopes.


In [ ]:
model = smf.ols('control ~ awareness + C(age)', data=sq2).fit(cov_type='HC3')
interaction = smf.ols('control ~ awareness * C(age)', data=sq2).fit(cov_type='HC3')
terms = [i for i, name in enumerate(interaction.params.index) if ':' in name]
interaction_p = float(interaction.wald_test(np.eye(len(interaction.params))[terms], scalar=True).pvalue)
b = float(model.params['awareness'])
lo, hi = model.conf_int().loc['awareness']
p = float(model.pvalues['awareness'])
rho, rho_p = stats.spearmanr(sq2.awareness, sq2.control)
print(f'N={len(sq2)}, awareness b={b:.3f}, 95% CI [{lo:.3f}, {hi:.3f}], p={p:.6f}, R²={model.rsquared:.3f}')
print(f'Age interaction p={interaction_p:.6g}; Spearman rho={rho:.3f}, p={rho_p:.6f}')
fig, ax = plt.subplots(figsize=(7, 2.5))
ax.errorbar(b, 0, xerr=[[b-lo], [hi-b]], fmt='o', capsize=5)
ax.axvline(0, color='grey', linestyle='--')
ax.set(yticks=[0], yticklabels=['Awareness'], xlabel='Control score difference per awareness step', ylim=(-.7, .7), title='Awareness after accounting for age')
plt.tight_layout(); plt.show()


## 4 Exploratory SEM
The SEM uses **awareness → Control** and **age groups → Control**. Control is a latent factor measured by the control, influence and reversed algorithm items. Age 18–24 is the reference. **H0:** The awareness path is zero. **H1:** It is nonzero.

This model treats answers as approximately continuous and uses MLW estimation. It assumes independent responses, one factor, uncorrelated item errors and comparable measurement across age groups. A model that converges has not proved those assumptions. The three-item control scale is weak, so this is an exploratory extra. The loading chart shows how well each item relates to Control. The separate interaction regression answers the age-moderation part of SQ2.


In [ ]:
import semopy
sem_rows = df[list(questions) + ['age']].dropna()
items = sem_rows[['control', 'influence', 'less_algorithm_control']].rename(columns={'control':'c1', 'influence':'c2', 'less_algorithm_control':'c3'})
alpha = 1.5 * (1 - items.var().sum() / items.sum(axis=1).var())
ages = pd.get_dummies(sem_rows.age).reindex(columns=['Under 18','25-34','35-44','45-54','65 and older'], fill_value=False)
ages.columns = ['under18', 'age25', 'age35', 'age45', 'age65']
sem_data = pd.concat([items, sem_rows[['awareness']], ages], axis=1).astype(float)
sem = semopy.Model('Control =~ c1 + c2 + c3\nControl ~ awareness + under18 + age25 + age35 + age45 + age65')
fit = sem.fit(sem_data)
assert fit.success, 'SEM did not converge'
estimates = sem.inspect(std_est=True, se_robust=True)
paths = estimates[(estimates.lval == 'Control') & (estimates.op == '~')]
loadings = estimates[(estimates.rval == 'Control') & estimates.lval.isin(['c1','c2','c3'])].set_index('lval')
variances = estimates[(estimates.op == '~~') & (estimates.lval == estimates.rval)]
sem_awareness = paths.loc[paths.rval == 'awareness'].iloc[0]
sem_b, sem_p = float(sem_awareness['Estimate']), float(sem_awareness['p-value'])
print(f'N={len(sem_data)}, alpha={alpha:.3f}, awareness b={sem_b:.3f}, p={sem_p:.6f}')
print('Negative variances:', int((variances.Estimate < 0).sum()))
display(paths[['rval','Estimate','Std. Err','p-value']].round(4))
ax = loadings.loc[['c1','c2','c3'], 'Est. Std'].astype(float).plot.bar(rot=0, figsize=(6, 3.5))
ax.set(ylim=(0, 1), xlabel='c1 control · c2 influence · c3 reversed algorithm', ylabel='Standardised loading', title=f'Candidate control scale: alpha = {alpha:.3f}')
plt.tight_layout(); plt.show()


## 5 Exploratory clustering
I use K-means to look for answer profiles in the four observed items. They are separate features, not a validated scale. I standardise them so each has equal variance weight. Age is left out and described afterwards.

I compare k = 2 to 10. Higher silhouette is better; lower Davies–Bouldin is better. I use the highest-silhouette solution as a provisional choice, not a confirmed optimum. K-means treats score gaps as equal and uses Euclidean distance. Cluster differences in these features are built into the method, not new hypothesis-test discoveries.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
cluster_data = df[list(questions)].dropna()
X = StandardScaler().fit_transform(cluster_data)
models, results = {}, []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=20, random_state=20261102).fit(X)
    models[k] = km
    results.append([k, km.inertia_, silhouette_score(X, km.labels_), davies_bouldin_score(X, km.labels_), np.bincount(km.labels_).min()])
metrics = pd.DataFrame(results, columns=['k','inertia','silhouette','DBI','smallest_cluster']).set_index('k')
best_k = int(metrics.silhouette.idxmax())
labelled = cluster_data.assign(cluster=models[best_k].labels_ + 1)
profiles = labelled.groupby('cluster')[list(questions)].mean()
profiles.insert(0, 'n', labelled.groupby('cluster').size())
print(f'N={len(cluster_data)}, provisional k={best_k}, silhouette={metrics.loc[best_k,"silhouette"]:.3f}')
display(profiles.round(3))
ax = profiles[['awareness','control']].plot.bar(rot=0, figsize=(8, 4))
ax.set(xlabel='Cluster label', ylabel='Mean score (1–5)', ylim=(0, 5.7), title=f'Awareness and control profiles at k = {best_k}')
ax.legend(['Awareness','Control'], ncol=2); plt.tight_layout(); plt.show()


## 6 What I found
The next cell gives the result in plain English. The main regression and the SEM use different measures of control and different complete-case samples, so their coefficients are not directly comparable.


In [ ]:
main_text = 'I did not find clear evidence of an overall awareness–control link after accounting for age.' if p >= .05 else 'I found evidence of an awareness–control link after accounting for age.'
age_text = 'The extra interaction test suggests that the link differs by age. Small age groups make this result less secure.' if interaction_p < .05 else 'The extra interaction test did not find clear evidence that the link differs by age.'
display(Markdown(f"""{main_text} **p = {p:.3f}**. {age_text}

The SEM gives an awareness path of **{sem_b:.3f}**, **p = {sem_p:.4f}**, but its control scale is weak (**alpha = {alpha:.3f}**). I therefore keep it exploratory.

The clustering gives **{best_k} provisional profiles** among the tested choices. This does not prove that real population groups exist. The appendix checks how much assignments change with random starts.

**Limits:** Awareness is not actual personalisation. Groups are small and unequal, missing answers are excluded, and independence depends on recruitment. The main model uses single items; SEM measurement is weak. No result shows cause and effect. The final multiple-testing correction and the course's two-path requirement still need group/lecturer agreement.
"""))


# Appendix — Amanda Luijendijk
These details support my main analysis. They are kept here so the main story stays short.

## A Ranking and scaling
Average ranks give tied answers the same rank. Min–max scores run from 0 to 1; z-scores show distance from the mean in standard deviations. Missing values stay missing. These transformations do not make answers normal, and they are not extra tests. The models above use original scores.


In [ ]:
scores = df[['awareness','control']]
ranks = scores.rank(method='average')
minmax = (scores - scores.min()) / (scores.max() - scores.min()).replace(0, np.nan)
z_scores = (scores - scores.mean()) / scores.std(ddof=0).replace(0, np.nan)
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, (name, values) in zip(axes, [('Average rank',ranks), ('Min max',minmax), ('Z score',z_scores)]):
    for column in scores:
        mapping = pd.DataFrame({'score':scores[column], 'value':values[column]}).dropna().groupby('score').value.mean()
        ax.plot(mapping.index, mapping.values, 'o-', label=column)
    ax.set(xlabel='Original score', ylabel=name, xticks=[1,2,3,4,5])
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()


## B Extra age comparisons
The rank test compares control distributions across age groups. It does not show which pairs differ; interpreting it only as a median test needs similar distribution shapes. A separate regression includes social media use frequency as categories. Neither test measures personalisation.

A temporary Bonferroni correction for three group questions is shown. It does not cover all exploratory tests. Agree the final testing family before confirmatory reporting.


In [ ]:
age_data = df[['age','control']].dropna()
h, age_p = stats.kruskal(*[g.control.to_numpy() for _, g in age_data.groupby('age')])
use_data = df[['age','control','social_use']].dropna()
use_model = smf.ols('control ~ C(age) + C(social_use)', data=use_data).fit(cov_type='HC3')
age_terms = [i for i, name in enumerate(use_model.params.index) if name.startswith('C(age)')]
use_p = float(use_model.wald_test(np.eye(len(use_model.params))[age_terms], scalar=True).pvalue)
print(f'Kruskal–Wallis: N={len(age_data)}, H={h:.3f}, p={age_p:.6g}; temporary corrected p={min(1,3*age_p):.6g}')
print(f'Age with use frequency: N={len(use_data)}, joint age p={use_p:.6g}, R²={use_model.rsquared:.3f}')
print('Temporary corrected main awareness p:', min(1, 3*p))
summary = age_data.groupby('age').control.agg(['count','mean','median']).reindex(age_order)
display(summary.round(3))
ax = summary['mean'].dropna().plot.bar(rot=25, figsize=(7, 3.5))
ax.set(xlabel='Age group', ylabel='Mean control score', ylim=(0,5), title='Control scores by age'); plt.tight_layout(); plt.show()


## C Age specific slopes and model checks
Dots show the fitted awareness slope within each age group, with unadjusted 95% HC3 confidence intervals. Small groups have uncertain estimates. The joint interaction p-value compares slopes together. SEM fit statistics do not validate weak measurement; the model's individual age paths are not a joint age test.


In [ ]:
from patsy import build_design_matrices
fig, ax = plt.subplots(figsize=(7, 3.5))
observed_ages = [a for a in age_order if (sq2.age == a).any()]
for i, age in enumerate(observed_ages):
    new = pd.DataFrame({'awareness':[1.,2.], 'age':[age,age]})
    design = np.asarray(build_design_matrices([interaction.model.data.design_info], new)[0])
    test = interaction.t_test(design[1] - design[0])
    slope = float(np.asarray(test.effect).item()); low, high = test.conf_int().ravel()
    ax.errorbar(slope, i, xerr=[[slope-low],[high-slope]], fmt='o', capsize=3)
ax.axvline(0, color='grey', linestyle='--')
ax.set(yticks=range(len(observed_ages)), yticklabels=[f'{a} (n={(sq2.age==a).sum()})' for a in observed_ages], xlabel='Control difference per awareness step', title='Exploratory age slopes')
ax.invert_yaxis(); plt.tight_layout(); plt.show()
print('Main regression design is full rank:', np.linalg.matrix_rank(model.model.exog) == model.model.exog.shape[1])
display(semopy.calc_stats(sem)[['DoF','chi2','chi2 p-value','CFI','TLI','RMSEA']].T.round(3))


## D Cluster choice stability and age profiles
Inertia falls as k increases, so it cannot choose k by itself. Compare the elbow, silhouette, Davies–Bouldin, group sizes and usefulness of the profiles. The highest silhouette is only a provisional rule among k = 2 to 10.

I refit that solution with five random seeds. Adjusted Rand Index of 1 means identical assignments after allowing for label changes. This checks random starts on the same sample, not new respondents. Age percentages are descriptions, not tests. DBSCAN and HDBSCAN were not fitted.


In [ ]:
display(metrics.round(4))
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, name in zip(axes, ['inertia','silhouette','DBI']):
    ax.plot(metrics.index, metrics[name], 'o-'); ax.set(xlabel='k', ylabel=name)
plt.tight_layout(); plt.show()
seeds = [11,22,33,44,55]
agreement = [adjusted_rand_score(models[best_k].labels_, KMeans(n_clusters=best_k, n_init=20, random_state=s).fit(X).labels_) for s in seeds]
ax = pd.Series(agreement, index=seeds).plot.bar(rot=0, figsize=(6, 3))
ax.set(xlabel='Random seed', ylabel='Adjusted Rand Index', ylim=(0,1.05), title='Same sample stability'); plt.tight_layout(); plt.show()
print('Agreement range:', round(min(agreement),3), 'to', round(max(agreement),3))
cluster_ages = df.loc[labelled.index, 'age'].fillna('Missing age')
age_counts = pd.crosstab(labelled.cluster, cluster_ages).reindex(columns=age_order+['Missing age'], fill_value=0)
display(age_counts)
display(age_counts.div(age_counts.sum(axis=1), axis=0).mul(100).round(1))


## Sources and final checks
Data: authorised group Responses.xlsx. Methods follow the course SEM and Clustering-with-Kmeans-DBSCAN-HDBSCAN examples, adapted to these survey questions. SEM reference: https://semopy.com/tutorial.html.

My two types of structural SEM paths are awareness → Control and age groups → Control. The lecturer still needs to confirm the course interpretation and the candidate measurement items. The weak control scale and provisional clusters are part of the findings, not problems solved by successful execution.

Prepared with Codex assistance. I should understand the code, acknowledge adapted sources and explain my own contribution. The other members' sections are outside this block.
